In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')
# Di chuyển vào thư mục code
os.chdir('/content/drive/My Drive/ColabNotebooks/ThachSanh')

Mounted at /content/drive


In [3]:
!uv pip install transformers==4.57.6

Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 246ms
Prepared 2 packages in 508ms
Uninstalled 2 packages in 360ms
Installed 2 packages in 54ms
 - huggingface-hub==1.5.0
 + huggingface-hub==0.36.2
 - transformers==5.0.0
 + transformers==4.57.6


In [4]:
!pip install -q unsloth==2025.9.7 trl==0.22.2 datasets peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.6/314.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [9]:
%%writefile train.py

from unsloth import FastLanguageModel
import os
import sys
import argparse
import wandb
import torch
import transformers
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments
from peft import AdaLoraConfig, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from datasets import load_dataset, Dataset
from trl import SFTTrainer, setup_chat_format, SFTConfig

# Tắt WandB để không bị hỏi đăng nhập (tránh treo máy)
os.environ["WANDB_DISABLED"] = "true"

os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3" # "0,1,2,3"
os.environ["OMP_NUM_THREAD"]="1"


def parse_args():
    parser = argparse.ArgumentParser(description="Parse Training Arguments")
    parser.add_argument('--pretrained_model_name_or_path', type=str, required=True,
                        help='Specify model name or path to set transformer backbone, required')
    parser.add_argument('--tokenizer_name_or_path', type=str, default=None,
                        help='Specify tokenizer name or path. Default None, will use model_name_or_path')
    parser.add_argument('--pretrained_peft_model_path', type=str, default=None,
                        help='Specify pretrained peft model path to load adapter, default None')
    parser.add_argument('--save_dir', type=str, default=None,
                        help='Specify save dir, default None')
    parser.add_argument('--train_data_path', type=str, required=True,
                        help='Specify training dataset')
    parser.add_argument('--use_lora', type=int, default=0, choices=[0,1],
                        help='Specify lora adapter training')
    parser.add_argument('--use_adalora', type=int, default=0, choices=[0,1],
                        help='Specify adalora adapter training')
    parser.add_argument('--init_r', type=int, default=12,
                        help='Specify initial AdaLoRA rank')
    parser.add_argument('--target_r', type=int, default=8,
                        help='Specify target AdaLoRA rank')
    parser.add_argument('--beta1', type=int, default=0.85,
                        help='Specify initial AdaLoRA beta1')
    parser.add_argument('--beta2', type=int, default=0.85,
                        help='Specify initial AdaLoRA beta2')
    parser.add_argument('--tinit', type=int, default=100,
                        help='Specify number of warmup steps for AdaLoRA wherein no pruning is performed')
    parser.add_argument('--tfinal', type=int, default=800,
                        help='Specify fix the resulting budget distribution and fine-tune the model for tfinal steps when using AdaLoRA')
    parser.add_argument('--deltaT', type=int, default=10,
                        help='Specify interval of steps for AdaLoRA to update rank')
    parser.add_argument('--lora_alpha',type=int, default=32,
                        help='Specify Lora alpha')
    parser.add_argument('--r', type=int, default=16,
                        help='Specify Lora rank')
    parser.add_argument('--lora_dropout', type=float, default=0.1,
                        help='Specify LoRA dropout',)
    parser.add_argument('--orth_reg_weight', type=float, default=0.1,
                        help="Specify orthogonal regularization weight")
    parser.add_argument('--is_kbit', type=bool, default=None,
                        help='Specify quatization load in')
    parser.add_argument('--load_kbit', type=int, default=None, choices=[4, 8],
                        help='Specify kbit training, choices [4, 8], default None')
    parser.add_argument('--torch_dtype', type=str, default="fp32", choices=['auto', 'fp32', 'fp16', 'bf16'],
                        help='Specify torch_dtype from [`auto`, `fp32`, `fp16`, `bf16`], default fp32')
    parser.add_argument('--fp16', type=int, default=0, choices=[0, 1],
                        help='Specify fp16, choices [0, 1], default None')
    parser.add_argument('--bf16', type=int, default=1, choices=[0, 1],
                        help='Specify bf16, choices [0, 1], default None')
    parser.add_argument('--num_train_epochs', type=int, default=2,
                        help='Specify epochs, default 2')
    parser.add_argument('--learning_rate', type=float, default=1e-4,
                        help='Specify learning rate')
    parser.add_argument('--lr_scheduler_type', type=str, default='cosine',
                        help='Specify learning scheduler type')
    parser.add_argument('--optim', type=str, default='adamw_8bit',
                        help='Specify optimizer')
    parser.add_argument('--warmup_steps', type=int, default=400,
                        help='Specify warm up step')
    parser.add_argument('--warmup_ratio', type=float, default=0.05,
                        help='Specify warm up ratio')
    parser.add_argument('--logging_steps', type=int, default=10,
                        help='Specify logging step')
    parser.add_argument('--save_steps', type=int, default=400,
                        help='Specify save step')
    parser.add_argument('--per_device_train_batch_size', type=int, default=4,
                        help='Specify batch size')
    parser.add_argument('--gradient_accumulation_steps', type=int, default=8,
                        help='Specify accumulation step')
    parser.add_argument('--is_training', type=bool, default=True,
                        help='Specify train mode')
    parser.add_argument('--do_eval', type=bool, default=False,
                        help='Specify eval mode')
    parser.add_argument('--report_to', type=str, default='wandb',
                        help='Specify report to environment')
    parser.add_argument('--weight_decay', type=float, default=0.0,
                        help='Specify weight decay to use')
    parser.add_argument('--device', type=str, default="cuda:0")

    args = parser.parse_args()

    return args

# ==========================================
# ĐÃ SỬA: Hàm chuẩn bị dữ liệu JSONL ChatML
# ==========================================
def prepare_train_dataset(train_data_path, tokenizer):
    # Dùng hàm load_dataset của HuggingFace để đọc thẳng file JSONL
    dataset = load_dataset("json", data_files=train_data_path, split="train")

    def format_chat_template(row):
        # Dữ liệu của bạn đã có sẵn cột "messages", chỉ cần áp template chuẩn của model
        row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
        return row

    training_data = dataset.map(
        format_chat_template,
        num_proc=4,
    )
    return training_data


class Args:
    # ĐÃ SỬA: Thay đổi base model thành Qwen 2.5 0.5B bản Instruct
    pretrained_model_name_or_path = "Qwen/Qwen2.5-0.5B-Instruct"

    tokenizer_name_or_path = None
    pretrained_peft_model_path = None

    # ĐÃ SỬA: Thư mục lưu và đường dẫn file dữ liệu
    save_dir = "./thachsanh_checkpoint/"
    train_data_path = "/content/drive/My Drive/ColabNotebooks/ThachSanh/train.jsonl"

    use_lora = 1
    use_adalora = 0
    init_r = 12
    target_r = 8
    beta1 = 0.85
    beta2 = 0.85
    tinit = 100
    tfinal = 800
    deltaT = 10
    lora_alpha = 32
    r = 16
    lora_r = 16
    lora_dropout = 0.1
    orth_reg_weight = 0.1
    is_kbit = True
    load_kbit = 4

    # Điều chỉnh dtype
    torch_dtype = torch.float16
    fp16 = True
    bf16 = False

    num_train_epochs = 3
    learning_rate = 4e-5
    lr_scheduler_type = "cosine"
    optim = "adamw_8bit"
    warmup_steps = 400
    warmup_ratio = 0.1
    logging_steps = 10
    save_steps = 2000
    per_device_train_batch_size = 1
    gradient_accumulation_steps = 1
    is_training = True
    do_eval = False
    report_to = "none"
    weight_decay = 0.0
    device = "cuda:0"


def main():
    # args = parse_args()
    args = Args()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    args.device = '''cuda:{}'''.format(local_rank)

    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        print(gpu_count)
    else:
        args.device = "cpu"
        gpu_count = 0

    base_model = args.pretrained_model_name_or_path

    if args.torch_dtype == 'fp32':
        args.torch_dtype = torch.float32
    elif args.torch_dtype == 'fp16':
        args.torch_dtype = torch.float16
    elif args.torch_dtype == 'bf16':
        args.torch_dtype = torch.bfloat16

    # load the model
    if args.is_kbit:
        bnb_config=BitsAndBytesConfig(
                        load_in_4bit=args.load_kbit == 4,
                        load_in_8bit=args.load_kbit == 8,
                        llm_int8_threshold=6.0,
                        llm_int8_has_fp16_weight=False,
                        bnb_4bit_compute_dtype=args.torch_dtype,
                        bnb_4bit_use_double_quant=True,
                        bnb_4bit_quant_type='nf4',
                    )

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model,
            dtype=args.torch_dtype,
            load_in_4bit=True,
            device_map=args.device
        )

    else:
        model = AutoModelForCausalLM.from_pretrained(
                    base_model,
                    device_map=args.device,
                    torch_dtype = args.torch_dtype,
                    trust_remote_code=True,
                    # attn_implementation="sdpa",
                )

    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    if args.pretrained_peft_model_path is not None:
        print(f'Load lora weight from {args.pretrained_peft_model_path}')
        model = PeftModel.from_pretrained(
            model,
            args.pretrained_peft_model_path,
            torch_dtype=torch.bfloat16 if args.is_kbit else args.torch_dtype,
            device_map=args.device,
            is_trainable=args.is_training
        )
    elif args.is_training and (args.use_adalora or args.use_lora):
        model = FastLanguageModel.get_peft_model(
            model,
            r=32,
            lora_alpha=64,
            target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth",
            bias="none",
        )
        model.print_trainable_parameters()

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = model.config.eos_token_id

    training_data = prepare_train_dataset(args.train_data_path, tokenizer)

    model = model.to(args.device)

    trainer = SFTTrainer(
        model=model,
        train_dataset=training_data,
        tokenizer=tokenizer,
        args=SFTConfig(
            output_dir=args.save_dir,
            lr_scheduler_type=args.lr_scheduler_type,
            warmup_ratio=args.warmup_ratio,
            per_device_train_batch_size=args.per_device_train_batch_size,
            gradient_accumulation_steps=args.gradient_accumulation_steps,
            num_train_epochs=args.num_train_epochs,
            learning_rate=args.learning_rate,
            bf16=bool(args.bf16),
            fp16=bool(args.fp16),
            optim=args.optim,
            logging_steps=args.logging_steps,
            save_steps=args.save_steps,
            do_eval=args.do_eval,
            weight_decay=args.weight_decay,
            dataset_text_field="text",
            packing=False,
            report_to=args.report_to,
            ddp_find_unused_parameters=False if gpu_count > 1 else None,
        ),
    )

    if args.pretrained_peft_model_path is not None:
        trainer.train(resume_from_checkpoint= args.pretrained_peft_model_path)
    else:
        trainer.train()

    print(f"Bắt đầu lưu model vào: {args.save_dir}")

    if not os.path.exists(args.save_dir):
        os.makedirs(args.save_dir)

    model.save_pretrained(args.save_dir)
    tokenizer.save_pretrained(args.save_dir)

    print("Đã lưu model thành công!")

if __name__ == '__main__':
    main()

Overwriting train.py


In [10]:
!python train.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-08 14:28:09.084716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772980089.121740    7395 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772980089.133377    7395 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772980089.163467    7395 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772980089.163509    7395 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [11]:
# --- CELL 1: CÀI ĐẶT & DỌN DẸP ---
!pip install -q unsloth pyvi nltk pandas

import torch
import gc
import os

# Dọn dẹp bộ nhớ cũ (quan trọng nếu bạn vừa train xong hoặc vừa bị lỗi)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("Đã dọn dẹp VRAM và cài xong thư viện!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.6 MB/s eta 0:00:00
Đã dọn dẹp VRAM và cài xong thư viện!


In [12]:
# --- CELL 2: LOAD MODEL ---
from unsloth import FastLanguageModel
import torch

# CẤU HÌNH ĐƯỜNG DẪN
# Nếu dùng Colab + Drive: "/content/drive/MyDrive/..."
# Nếu dùng Kaggle: "/kaggle/working/checkpoint/checkpoint-5500"
checkpoint_path = "/content/drive/My Drive/ColabNotebooks/ThachSanh/thachsanh_checkpoint/checkpoint-2925"

print(f"Đang load model từ: {checkpoint_path}")

# Load model với Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = checkpoint_path,
    max_seq_length = 1024,
    dtype = None,       # Auto detection (float16/bfloat16)
    load_in_4bit = True # Load 4bit để tiết kiệm VRAM
)

# Chuyển sang chế độ Inference (nhanh hơn 2x)
FastLanguageModel.for_inference(model)

# Xác định device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model đã load thành công trên thiết bị: {device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Đang load model từ: /content/drive/My Drive/ColabNotebooks/ThachSanh/thachsanh_checkpoint/checkpoint-2925
==((====))==  Unsloth 2025.9.7: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.9.7 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model đã load thành công trên thiết bị: cuda


In [28]:
# ==========================================
# HÀM TRÒ CHUYỆN VỚI THẠCH SANH
# ==========================================
def chat_with_thach_sanh(user_question, context_text=""):
    # 1. Chọn System Prompt đúng với dữ liệu lúc train
    if context_text.strip() == "":
        system_prompt = "Bạn là Thạch Sanh trong truyện cổ tích Việt Nam. Hãy trò chuyện thân thiện với người dùng và xưng 'tôi'."
    else:
        system_prompt = f"Bạn là Thạch Sanh trong truyện cổ tích Việt Nam. Hãy trả lời câu hỏi của người dùng, xưng 'tôi', dựa trên bối cảnh sau: {context_text}"

    # 2. Đóng gói thành mảng messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]

    # 3. Dùng tokenizer để bọc thẻ ChatML tự động
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True, # BẮT BUỘC TRUE: Báo cho model biết đến lượt nó trả lời
        return_tensors="pt"
    ).to(device)

    # 4. Chạy suy luận (Inference) sinh ra câu trả lời
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512, # Số từ tối đa model được phép nói
        use_cache=True,
        temperature=0.3,    # Để thấp (0.1 - 0.3) để model trả lời chính xác, bớt bịa chuyện
        top_p=0.9,
    )

    # 5. Giải mã output từ các con số thành chữ (và cắt bỏ phần câu hỏi ban đầu đi)
    response_tokens = outputs[0][inputs.shape[1]:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=True)

    return response

# ==========================================
# TEST HỆ THỐNG
# ==========================================
print("\n" + "="*50)
print("🤖 BẮT ĐẦU TEST MÔ HÌNH THẠCH SANH")
print("="*50 + "\n")

# --- KỊCH BẢN 1: CHAT THÔNG THƯỜNG (KHÔNG BỐI CẢNH) ---
print("👉 TEST 1: HỎI THÔNG TIN CƠ BẢN (KHÔNG CONTEXT)")
q1 = "Anh nghĩ sao về Vua Hùng?"
print(f"👤 Người dùng: {q1}")
print(f"🗡️ Thạch Sanh: {chat_with_thach_sanh(q1)}\n")

# --- KỊCH BẢN 2: HỎI CÓ BỐI CẢNH (RAG) ---
print("👉 TEST 2: ĐỌC HIỂU BỐI CẢNH (RAG CONTEXT)")
ctx = "Thạch Sanh cứu thái tử con vua Thủy Tề ra và được mời về thủy cung để vua cha được đền ơn. Vua Thủy Tề mừng lắm, tặng Thạch Sanh vô số vàng bạc châu báu, nhưng chàng đều từ chối không nhận, chỉ nhận lấy một cây đàn. Xong rồi từ giã vua và thái tử, lên trần gian, về chốn cũ ở gốc đa."
q2 = "Cuộc sống của anh có thay đổi gì sau chuyến đi thủy cung không?"
print(f"👤 Người dùng: {q2}")
print(f"🗡️ Thạch Sanh: {chat_with_thach_sanh(q2, ctx)}\n")


🤖 BẮT ĐẦU TEST MÔ HÌNH THẠCH SANH

👉 TEST 1: HỎI THÔNG TIN CƠ BẢN (KHÔNG CONTEXT)
👤 Người dùng: Anh nghĩ sao về Vua Hùng?
🗡️ Thạch Sanh: Vua Hùng là người cứu tôi thoát khỏi hang sâu, giúp tôi trở lại đất nước. Anh ấy là người con trai của tôi, nên tôi rất cảm tạ.

👉 TEST 2: ĐỌC HIỂU BỐI CẢNH (RAG CONTEXT)
👤 Người dùng: Cuộc sống của anh có thay đổi gì sau chuyến đi thủy cung không?
🗡️ Thạch Sanh: Tôi vẫn giữ nguyên hình dáng cũ, lại thêm một phần mới vào vì đã từ bỏ tất cả những thứ mà vua Thủy Tề đã tặng.



In [ ]:
from huggingface_hub import login

# 1. Đăng nhập (Thay TOKEN_CUA_BAN bằng token vừa tạo)
login(token="Nhap token cua ban")

# 2. Đặt tên model trên HF (username/tên-model)
# Ví dụ: "nguyenvana/sailor2-viet-khmer-lora"
repo_name = "pqanh149/Qwen2.5-0.5B-Instruct_ThachSanh1"

print(f"Đang đẩy model lên: {repo_name}...")

# 3. Lưu và đẩy lên Hub (chỉ đẩy Adapter LoRA - rất nhẹ)
model.push_to_hub(repo_name, use_temp_dir= True)
tokenizer.push_to_hub(repo_name, use_temp_dir= True)

print("Đã xong! Lần sau chỉ cần load từ tên repo này.")

Đang đẩy model lên: pqanh149/Qwen2.5-0.5B-Instruct_ThachSanh1...


README.md:   0%|          | 0.00/613 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  555kB / 70.4MB            

Saved model to https://huggingface.co/pqanh149/Qwen2.5-0.5B-Instruct_ThachSanh1


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcmmzjo44/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

Đã xong! Lần sau chỉ cần load từ tên repo này.
